# 18. Feature Engineering for All Splits
#
# 분할 파일(train_raw, val_tune_raw, val_calib_raw, test_raw) 중 **선택한 것만**
# 파생변수를 생성하고, 2014-04-01 이전 데이터를 제거한 후 저장합니다.
#
# 출력 컬럼 (총 20개):
#   serial_number, date, failure + 17개 파생변수
#
# 실행 대상은 아래 셀의 `RUN_SPLITS` 에서 지정합니다.
#
# - `RUN_SPLITS = None` → **raw 파일이 있는 분할만** 자동 처리 (없는 것만 건너뜀)
# - `RUN_SPLITS = ["val_calib"]` → 지정 분할 **필수** (raw 없으면 즉시 오류, 조용히 skip 안 함)
# - raw 경로를 직접 줄 때: `INPUT_OVERRIDES`


In [1]:
import sys
import duckdb
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import os
import gc
import math
from pathlib import Path

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, BASE_DIR)
from config.path_utils import DATA_ROOT, FE_SPLIT_REL

INPUT_DIR = DATA_ROOT / FE_SPLIT_REL
OUTPUT_DIR = DATA_ROOT / '06_hyperparameter_tuning'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 분할 카탈로그: 키 → (입력 raw, 출력)
SPLIT_CATALOG = {
    "train":     ("train_raw.parquet",     "train.parquet"),
    "val_tune":  ("val_tune_raw.parquet",  "val_tune.parquet"),
    "val_calib": ("val_calib_raw.parquet", "val_calib.parquet"),
    "test":      ("test_raw.parquet",      "test.parquet"),
}

# ── 실행 대상 선택 ───────────────────────────────────────────
# None 또는 [] → raw 가 있는 분할만 자동 실행
# 리스트 지정 → 해당 분할 raw 필수 (없으면 즉시 오류)
RUN_SPLITS = None  # 예: ["val_calib"], ["val_tune", "val_calib"]

INPUT_OVERRIDES: dict[str, str] = {
    # "val_calib": r"C:\path\to\val_calib_raw.parquet",
}
RAW_SEARCH_DIRS = [INPUT_DIR, DATA_ROOT / "06a_feature_engineering"]

CUT_DATE = "2014-04-01"  # 이 날짜 미만(2014-03-31 이하) 데이터 제거


def _find_raw(raw_name: str, split_key: str) -> Path | None:
    if split_key in INPUT_OVERRIDES:
        p = Path(INPUT_OVERRIDES[split_key]).expanduser()
        return p if p.is_file() else None
    for d in RAW_SEARCH_DIRS:
        p = Path(d) / raw_name
        if p.is_file():
            return p.resolve()
    return None


def print_split_inventory() -> None:
    print(f"\n[분할 데이터 현황] INPUT_DIR = {INPUT_DIR}, OUTPUT_DIR = {OUTPUT_DIR}")
    print(f"  {'split':<10} {'raw':^8} {'output':^8}  파일")
    for k, (raw, out) in SPLIT_CATALOG.items():
        raw_p = _find_raw(raw, k)
        out_p = OUTPUT_DIR / out
        raw_st = "OK" if raw_p else "MISSING"
        out_st = "OK" if out_p.is_file() else "—"
        print(f"  {k:<10} {raw_st:^8} {out_st:^8}  {raw} → {out}")
        if raw_p and raw_p.parent != INPUT_DIR:
            print(f"             └─ raw 위치: {raw_p}")


def build_tasks(run_splits: list[str] | None, *, fail_on_missing: bool) -> list[dict]:
    keys = list(SPLIT_CATALOG) if not run_splits else list(run_splits)
    unknown = set(keys) - set(SPLIT_CATALOG)
    if unknown:
        raise ValueError(
            f"Unknown RUN_SPLITS keys: {sorted(unknown)}. "
            f"Choose from: {list(SPLIT_CATALOG)}"
        )

    tasks, missing = [], []
    for k in keys:
        raw, out = SPLIT_CATALOG[k]
        in_p = _find_raw(raw, k)
        if in_p is None:
            missing.append((k, raw, INPUT_DIR / raw))
            continue
        tasks.append({
            "key": k, "raw": raw, "out": out,
            "in_path": in_p, "out_path": OUTPUT_DIR / out,
        })

    if missing:
        print("\n[raw 없음]")
        for k, raw, expected in missing:
            print(f"  [{k}] 기대: {expected}")
        if fail_on_missing:
            raise FileNotFoundError(
                "선택한 분할의 raw 파일이 없습니다.\n"
                "  · README '데이터 분할 후' parquet → DATA_DIR 배치\n"
                "  · 또는 notebooks/03_data_splitting.ipynb 실행\n"
                "  · 또는 INPUT_OVERRIDES['split_key'] = r'절대경로'\n"
                f"INPUT_DIR = {INPUT_DIR}"
            )

    if not tasks:
        raise FileNotFoundError(
            "처리할 분할이 없습니다. 위 [분할 데이터 현황]을 확인하세요."
        )
    return tasks


print_split_inventory()
FAIL_ON_MISSING = bool(RUN_SPLITS)
TASKS = build_tasks(RUN_SPLITS, fail_on_missing=FAIL_ON_MISSING)

print("\n[이번 실행 — 처리 예정]" + (" (raw 있는 것만)" if not RUN_SPLITS else ""))
for t in TASKS:
    print(f"  [{t['key']}] {t['in_path']} → {t['out_path']}")


[분할 데이터 현황] INPUT_DIR = C:\Workspace\06_ML_projdect\26_1_COIN\data2\03_splitting, OUTPUT_DIR = C:\Workspace\06_ML_projdect\26_1_COIN\data2\06_hyperparameter_tuning
  split        raw     output   파일
  train         OK       —      train_raw.parquet → train.parquet
  val_tune      OK       —      val_tune_raw.parquet → val_tune.parquet
  val_calib     OK       —      val_calib_raw.parquet → val_calib.parquet
  test          OK       —      test_raw.parquet → test.parquet

[이번 실행 — 처리 예정] (raw 있는 것만)
  [train] C:\Workspace\06_ML_projdect\26_1_COIN\data2\03_splitting\train_raw.parquet → C:\Workspace\06_ML_projdect\26_1_COIN\data2\06_hyperparameter_tuning\train.parquet
  [val_tune] C:\Workspace\06_ML_projdect\26_1_COIN\data2\03_splitting\val_tune_raw.parquet → C:\Workspace\06_ML_projdect\26_1_COIN\data2\06_hyperparameter_tuning\val_tune.parquet
  [val_calib] C:\Workspace\06_ML_projdect\26_1_COIN\data2\03_splitting\val_calib_raw.parquet → C:\Workspace\06_ML_projdect\26_1_COIN\data2\06_hype

## SQL 빌더: 27개 파생변수 + date 필터


In [2]:
def build_chunk_sql(in_path, serial_str, cut_date):
    """
    청크 단위 SQL:
    - raw 파일로부터 diff 계산
    - 윈도우 함수로 파생변수 생성
    - cut_date 이전 데이터 제거
    - EWMA 대상 원본값(_r190, _r194)도 함께 반환
    """
    return f"""
WITH raw AS (
    SELECT
        serial_number, failure,
        CAST(date AS DATE) AS dt,
        smart_5_raw, smart_184_raw, smart_187_raw, smart_197_raw, smart_198_raw,
        smart_9_raw, smart_199_raw, smart_241_raw, smart_242_raw,
        timeout_total, total_reads, total_seeks,
        smart_190_raw, smart_194_raw,
        smart_183_raw, -- Raw SMART 183
        smart_5_raw   - LAG(smart_5_raw)   OVER w AS s5_diff,
        smart_187_raw - LAG(smart_187_raw) OVER w AS s187_diff,
        smart_197_raw - LAG(smart_197_raw) OVER w AS s197_diff,
        smart_198_raw - LAG(smart_198_raw) OVER w AS s198_diff,
        smart_199_raw - LAG(smart_199_raw) OVER w AS s199_diff,
        smart_241_raw - LAG(smart_241_raw) OVER w AS s241_diff,
        smart_242_raw - LAG(smart_242_raw) OVER w AS s242_diff,
        timeout_total - LAG(timeout_total) OVER w AS timeout_total_diff,
        total_reads   - LAG(total_reads)   OVER w AS total_reads_diff,
        total_seeks   - LAG(total_seeks)   OVER w AS total_seeks_diff,
        smart_9_raw   - LAG(smart_9_raw)   OVER w AS s9_diff
    FROM read_parquet('{in_path}')
    WHERE serial_number IN (SELECT unnest([{serial_str}]))
    WINDOW w AS (PARTITION BY serial_number ORDER BY date)
),
d2 AS (
    SELECT *,
        COALESCE(total_seeks_diff, 0) - LAG(COALESCE(total_seeks_diff, 0)) OVER w AS d_seeks,
        COALESCE(total_reads_diff, 0) - LAG(COALESCE(total_reads_diff, 0)) OVER w AS d_reads,
        COALESCE(s242_diff, 0)        - LAG(COALESCE(s242_diff, 0))        OVER w AS d_s242
    FROM raw
    WINDOW w AS (PARTITION BY serial_number ORDER BY dt)
),
calc AS (
    SELECT
        serial_number,
        dt           AS date,
        failure,

        -- 직접 raw 컬럼
        CAST(smart_198_raw AS FLOAT)  AS smart_198_raw,
        CAST(smart_242_raw AS FLOAT)  AS smart_242_raw,
        CAST(smart_241_raw AS FLOAT)  AS smart_241_raw,
        CAST(smart_9_raw   AS FLOAT)  AS smart_9_raw,
        CAST(smart_184_raw AS FLOAT)  AS smart_184_raw,
        CAST(smart_187_raw AS FLOAT)  AS smart_187_raw,
        CAST(smart_183_raw AS FLOAT)  AS smart_183_raw,

        -- diff 컬럼
        CAST(COALESCE(s5_diff, 0)          AS FLOAT) AS s5_diff,
        CAST(COALESCE(s241_diff, 0)        AS FLOAT) AS s241_diff,
        CAST(COALESCE(total_seeks_diff, 0) AS FLOAT) AS total_seeks_diff,

        -- reallocated_pending_ratio: (smart_197_raw + 1.0) / (smart_5_raw + 1.0)
        CAST((smart_197_raw + 1.0) / (smart_5_raw + 1.0) AS FLOAT) AS reallocated_pending_ratio,

        -- total_seeks_14d_mean: 14d mean seeks
        CAST(AVG(COALESCE(total_seeks_diff, 0.0)) OVER w14 AS FLOAT) AS total_seeks_14d_mean,

        -- workload_intensity: (smart_241_raw + smart_242_raw + 1.0) / (smart_9_raw + 1.0)
        CAST((smart_241_raw + smart_242_raw + 1.0) / (smart_9_raw + 1.0) AS FLOAT) AS workload_intensity,

        -- age_weighted_workload: ln(|s241_diff + s242_diff| + 1) * ln(smart_9_raw + 1)
        CAST(LN(ABS(COALESCE(s241_diff, 0) + COALESCE(s242_diff, 0)) + 1.0) * LN(COALESCE(smart_9_raw, 0.0) + 1.0) AS FLOAT) AS age_weighted_workload,

        -- multi_error_count: 5대 변수 diff > 0 개수
        CAST(
            CAST(COALESCE(s5_diff,0)>0 AS INT)
          + CAST(COALESCE(s187_diff,0)>0 AS INT)
          + CAST(COALESCE(s197_diff,0)>0 AS INT)
          + CAST(COALESCE(s198_diff,0)>0 AS INT)
          + CAST(COALESCE(timeout_total_diff,0)>0 AS INT)
        AS FLOAT)                                    AS multi_error_count,

        -- s187_28d_sum
        CAST(SUM(COALESCE(s187_diff,0)) OVER w28 AS FLOAT) AS s187_28d_sum,

        -- total_seeks_28d_asfd: SUM(|d_seeks|) 28d
        CAST(SUM(ABS(COALESCE(d_seeks,0))) OVER w28 AS FLOAT) AS total_seeks_28d_asfd,
        
        -- s242_28d_asfd: SUM(|d_s242|) 28d
        CAST(SUM(ABS(COALESCE(d_s242,0))) OVER w28 AS FLOAT) AS s242_28d_asfd,

        -- 온도 및 통계
        CAST(MAX(smart_194_raw) OVER w14 AS FLOAT)                        AS s194_14d_max,

        -- days_since_first
        CASE
          WHEN MIN(CASE WHEN smart_5_raw>0 THEN dt END) OVER w IS NULL THEN -1
          ELSE CAST(date_diff('day',
               CAST(MIN(CASE WHEN smart_5_raw>0 THEN dt END) OVER w AS DATE), dt)
               AS INTEGER)
        END                                          AS s5_days_since_first,

        CASE
          WHEN MIN(CASE WHEN smart_187_raw>0 THEN dt END) OVER w IS NULL THEN -1
          ELSE CAST(date_diff('day',
               CAST(MIN(CASE WHEN smart_187_raw>0 THEN dt END) OVER w AS DATE), dt)
               AS INTEGER)
        END                                          AS s187_days_since_first,

        -- EWMA 원본
        smart_190_raw AS _r190,
        smart_194_raw AS _r194

    FROM d2
    WINDOW
        w       AS (PARTITION BY serial_number ORDER BY dt),
        w14     AS (PARTITION BY serial_number ORDER BY dt ROWS BETWEEN 13 PRECEDING AND CURRENT ROW),
        w28     AS (PARTITION BY serial_number ORDER BY dt ROWS BETWEEN 27 PRECEDING AND CURRENT ROW)
)
SELECT * FROM calc
WHERE date >= CAST('{cut_date}' AS DATE)
ORDER BY serial_number, date
"""


## 처리 함수 정의


In [3]:
# 최종 출력 컬럼 순서 
FINAL_COLS = [
    "serial_number",
    "date",
    "failure",
    "reallocated_pending_ratio",
    "total_seeks_14d_mean",
    "s194_14d_max",
    "smart_187_raw",
    "s187_days_since_first",
    "workload_intensity",
    "s241_diff",
    "smart_183_raw",
    "total_seeks_28d_asfd",
    "smart_9_raw",
    "smart_241_raw",
    "s187_28d_sum",
    "smart_184_raw",
    "smart_198_raw",
    "multi_error_count",
    "age_weighted_workload",
    "smart_242_raw"
]


def process_file(in_file, out_file, cut_date="2014-04-01", chunk_size=1000):
    in_path = in_file.replace("\\", "/")

    print(f">>> Processing: {os.path.basename(in_file)}")

    con = duckdb.connect()

    # 시리얼 목록 스캔
    serials = con.execute(
        f"SELECT DISTINCT serial_number FROM read_parquet('{in_path}')"
    ).df()["serial_number"].tolist()

    total_chunks = math.ceil(len(serials) / chunk_size)
    print(f"  총 {len(serials):,}개 디스크, {total_chunks}개 청크")

    writer = None

    for chunk_idx, i in enumerate(range(0, len(serials), chunk_size)):
        chunk_serials = serials[i: i + chunk_size]
        serial_str = ", ".join(
            [f"'{s}'" if isinstance(s, str) else str(s) for s in chunk_serials]
        )

        sql = build_chunk_sql(in_path, serial_str, cut_date)
        df  = con.query(sql).df()

        if df.empty:
            continue

        df.fillna(0.0, inplace=True)
        df.sort_values(["serial_number", "date"], inplace=True)
        df.reset_index(drop=True, inplace=True)


        # 최종 컬럼 선택
        existing = [c for c in FINAL_COLS if c in df.columns]
        df = df[existing]

        # float64 -> float32 압축
        f64_cols = df.select_dtypes(include=["float64"]).columns
        df[f64_cols] = df[f64_cols].astype("float32")

        table = pa.Table.from_pandas(df, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(out_file, table.schema, compression="zstd")
        writer.write_table(table)

        del df, table
        gc.collect()
        print(f"  [{chunk_idx+1}/{total_chunks}] 완료", end="\r")

    if writer:
        writer.close()
    con.close()
    print(f"\n  ✅ 저장: {out_file}\n")


## 실행


In [4]:
# TASKS 에 담긴 분할만 처리 (raw 경로는 이미 검증됨)
for t in TASKS:
    in_f  = str(t["in_path"])
    out_f = str(t["out_path"])
    print(f">>> Processing [{t['key']}]: {t['raw']}")

    if os.path.exists(out_f):
        os.remove(out_f)

    process_file(in_f, out_f, cut_date=CUT_DATE)

print(f"🎉 완료! 처리 {len(TASKS)}개: {[t['key'] for t in TASKS]}")


>>> Processing [train]: train_raw.parquet
>>> Processing: train_raw.parquet
  총 50,517개 디스크, 51개 청크
  [51/51] 완료
  ✅ 저장: C:\Workspace\06_ML_projdect\26_1_COIN\data2\06_hyperparameter_tuning\train.parquet

>>> Processing [val_tune]: val_tune_raw.parquet
>>> Processing: val_tune_raw.parquet
  총 8,419개 디스크, 9개 청크
  [9/9] 완료
  ✅ 저장: C:\Workspace\06_ML_projdect\26_1_COIN\data2\06_hyperparameter_tuning\val_tune.parquet

>>> Processing [val_calib]: val_calib_raw.parquet
>>> Processing: val_calib_raw.parquet
  총 8,453개 디스크, 9개 청크
  [9/9] 완료
  ✅ 저장: C:\Workspace\06_ML_projdect\26_1_COIN\data2\06_hyperparameter_tuning\val_calib.parquet

>>> Processing [test]: test_raw.parquet
>>> Processing: test_raw.parquet
  총 16,819개 디스크, 17개 청크
  [17/17] 완료
  ✅ 저장: C:\Workspace\06_ML_projdect\26_1_COIN\data2\06_hyperparameter_tuning\test.parquet

🎉 완료! 처리 4개: ['train', 'val_tune', 'val_calib', 'test']


## 5. 엄밀한 피처 엔지니어링 결과 및 스키마 무결성 검증 테스트 (Verification Tests)

변환된 각 분할 데이터셋의 파일 존재 여부, 스키마 동일성(21개 컬럼), 2014-04-01 이전 데이터의 완벽한 제거 여부를 검증하여 피처 엔지니어링 과정의 정합성을 입증합니다.

In [5]:
import duckdb
import os

con = duckdb.connect()

print("🔍 [6-A단계 무결성 검증] 시작...")

try:
    # 1. 파일 존재 확인 검증
    print("Test 1: 출력 파일 존재 확인 검증")
    for t in TASKS:
        out_path = t["out_path"]
        assert out_path.is_file(), f"오류: 출력 파일 {out_path}이 생성되지 않았습니다"
    print("  -> [PASS] 출력 파일 존재 확인.")

    # 2. 스키마 컬럼 개수 및 구성 검증
    print("Test 2: 컬럼 스키마 유효성 검증")
    for t in TASKS:
        out_f = str(t["out_path"]).replace("\\", "/")
        cols = con.execute(f"SELECT * FROM read_parquet('{out_f}') WHERE 1=0").df().columns.tolist()
        assert len(cols) == 20, f"오류: {t['out']}의 컬럼 수({len(cols)}개)가 20개가 아닙니다!"
        
        # 필수 메타 컬럼 확인
        for key_col in ["serial_number", "date", "failure"]:
            assert key_col in cols, f"오류: 필수 컬럼 '{key_col}'이 누락되었습니다"
    print("  -> [PASS] 20개 컬럼 스키마 및 메타 컬럼 통합성 확인.")

    # 3. 일자 컷오프(2014-04-01 이후) 검증
    print("Test 3: 일자 컷오프(2014-04-01) 기준 준수 여부 검증")
    for t in TASKS:
        out_f = str(t["out_path"]).replace("\\", "/")
        min_date = con.execute(f"SELECT MIN(date) FROM read_parquet('{out_f}')").fetchone()[0]
        min_date_str = min_date.strftime('%Y-%m-%d') if hasattr(min_date, 'strftime') else str(min_date)
        assert min_date_str >= '2014-04-01', f"오류: {t['out']}에 2014-04-01 이전 데이터가 포함되어 있습니다: {min_date_str}"
    print("  -> [PASS] 일자 컷오프 기준 준수 확인.")

    # ── 강화 4: row 수 > 0 검증 ──
    print("Test 4: 각 파일 row 수 > 0 검증")
    for t in TASKS:
        out_f = str(t["out_path"]).replace("\\", "/")
        cnt = con.execute(f"SELECT COUNT(*) FROM read_parquet('{out_f}')").fetchone()[0]
        assert cnt > 0, f"오류: {t['out']} 파일이 비어 있습니다"
    print("  -> [PASS] 모든 파일에 데이터 존재.")

    # ── 강화 5: NaN/Inf 검증 ──
    print("Test 5: NaN/Inf 잔존 검증")
    for t in TASKS:
        out_f = str(t["out_path"]).replace("\\", "/")
        all_cols = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{out_f}')").fetchdf()
        numeric_cols = [r['column_name'] for _, r in all_cols.iterrows() if r['column_name'] not in ('serial_number', 'date') and 'VARCHAR' not in r['column_type']]
        nan_checks = [f"SUM(CASE WHEN isnan(\"{nc}\") OR isinf(\"{nc}\") THEN 1 ELSE 0 END)" for nc in numeric_cols]
        if nan_checks:
            nan_results = con.execute(f"SELECT {', '.join(nan_checks)} FROM read_parquet('{out_f}')").fetchone()
            bad_cols = [numeric_cols[j] for j, v in enumerate(nan_results) if v and v > 0]
            assert len(bad_cols) == 0, f"오류: {t['out']}에 NaN/Inf 잔존 컬럼: {bad_cols}"
    print("  -> [PASS] NaN/Inf 없음 확인.")

    # ── 강화 6: failure 값 범위 검증 ──
    print("Test 6: failure 컬럼 값 범위 {0,1} 검증")
    for t in TASKS:
        out_f = str(t["out_path"]).replace("\\", "/")
        fv = con.execute(f"SELECT DISTINCT failure FROM read_parquet('{out_f}')").fetchall()
        fset = set(r[0] for r in fv)
        assert fset.issubset({0, 1}), f"오류: {t['out']} failure에 0/1 이외 값: {fset}"
    print("  -> [PASS] failure 값 범위 정상.")

    # ── 강화 7: 스키마 일관성 (모든 출력 파일 동일 컬럼) 검증 ──
    print("Test 7: 모든 출력 파일 간 스키마 일관성 검증")
    ref_cols = None
    for t in TASKS:
        out_f = str(t["out_path"]).replace("\\", "/")
        cols = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{out_f}')").fetchdf()['column_name'].tolist()
        if ref_cols is None:
            ref_cols = cols
        else:
            assert cols == ref_cols, f"오류: {t['out']} 스키마가 첫 번째 출력과 불일치!"
    print("  -> [PASS] 모든 출력 파일 스키마 일관성 확인.")

    print("\n✅ [6-A단계 통합성 검증 완료] 모든 정밀 테스트 조건을 만족합니다 (7/7 PASS)")
finally:
    con.close()


🔍 [6-A단계 무결성 검증] 시작...
Test 1: 출력 파일 존재 확인 검증
  -> [PASS] 출력 파일 존재 확인.
Test 2: 컬럼 스키마 유효성 검증
  -> [PASS] 20개 컬럼 스키마 및 메타 컬럼 통합성 확인.
Test 3: 일자 컷오프(2014-04-01) 기준 준수 여부 검증
  -> [PASS] 일자 컷오프 기준 준수 확인.
Test 4: 각 파일 row 수 > 0 검증
  -> [PASS] 모든 파일에 데이터 존재.
Test 5: NaN/Inf 잔존 검증
  -> [PASS] NaN/Inf 없음 확인.
Test 6: failure 컬럼 값 범위 {0,1} 검증
  -> [PASS] failure 값 범위 정상.
Test 7: 모든 출력 파일 간 스키마 일관성 검증
  -> [PASS] 모든 출력 파일 스키마 일관성 확인.

✅ [6-A단계 통합성 검증 완료] 모든 정밀 테스트 조건을 만족합니다 (7/7 PASS)
